# RoadGuard: Comprehensive Experimental Evaluation
**Project Context**: Master's Thesis in Computer Science Engineering (Sapienza University of Rome)
**Dataset**: Thessaloniki Road Quality Dataset & Figshare Pothole Dataset
**Objective**: This notebook executes the complete validation pipeline, including:
1. Single-modality performance (IMU vs Vision).
2. Late-fusion synergy gains.
3. Federated Learning (FedAvg) convergence.
4. Differential Privacy (DP) Utility-Privacy tradeoffs.

---

## 1. Environment Setup
Installing core dependencies and mounting storage.

In [ ]:
!pip install ultralytics kaggle scikit-learn matplotlib pandas numpy tensorflow -q
from google.colab import drive
import os, shutil, glob
from IPython.display import Image, display
import pandas as pd

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/RoadGuard_Thesis_Evaluation'
os.makedirs(SAVE_DIR, exist_ok=True)

# Clone or Update Repository
if not os.path.exists('/content/RoadGuard'):
    !git clone https://github.com/antoninofoti/RoadGuard.git /content/RoadGuard
else:
    %cd /content/RoadGuard
    !git pull
    %cd /content

print('Environment and Repository ready.')

## 2. Dataset Acquisition & Preprocessing
Downloading Thessaloniki dataset and normalizing IMU signals.

In [ ]:
# Auth: Requires kaggle.json
from google.colab import files
if not os.path.exists('kaggle.json'):
    print("Action Required: Please upload your kaggle.json file")
    uploaded = files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

!kaggle datasets download nickkotarelas/road-quality-dataset -p /content/data/thessaloniki/ --unzip

# IMU Normalization
csv_files = glob.glob('/content/data/thessaloniki/**/*.csv', recursive=True)
if csv_files:
    df = pd.read_csv(csv_files[0])
    df.columns = [c.strip().lower() for c in df.columns]
    # Align with RoadGuard internal schema
    COLUMN_MAP = {'x_acc':'acc_x', 'y_acc':'acc_y', 'z_acc':'acc_z'}
    df.rename(columns=COLUMN_MAP, inplace=True)
    NORMALIZED_CSV = '/content/data/thessaloniki/imu_normalized.csv'
    df.to_csv(NORMALIZED_CSV, index=False)
    print(f'IMU data normalized: {NORMALIZED_CSV}')

## 3. Vision Branch: Training & TFLite Export
Fine-tuning YOLOv8n and exporting to INT8 Quantized TFLite for Android assets.

In [ ]:
# Download training dataset
!wget -q 'https://figshare.com/ndownloader/articles/21214400/versions/3' -O /content/pothole_dataset.zip
!unzip -q /content/pothole_dataset.zip -d /content/data/pothole_dataset/

# Generate YOLOv8 configuration
yaml_content = """
path: /content/data/pothole_dataset
train: images/train
val:   images/val
names:
  0: pothole
"""
with open('/content/pothole_yolo.yaml', 'w') as f:
    f.write(yaml_content.strip())

from ultralytics import YOLO
model = YOLO('yolov8n.pt')

# Run training
model.train(data='/content/pothole_yolo.yaml', epochs=50, imgsz=640, project='/content/runs', name='roadguard_v1')

# --- NEW: TFLite INT8 Export ---
print('Exporting to TFLite INT8...')
model.export(format='tflite', int8=True)
TFLITE_EXPORT_PATH = '/content/runs/roadguard_v1/weights/best_int8.tflite'
if os.path.exists(TFLITE_EXPORT_PATH):
    shutil.copy(TFLITE_EXPORT_PATH, '/content/yolov8n_pothole.tflite')
    print('Model ready for Android.')
else:
    print('INT8 export not found, check ultralytics export log.')

## 4. Multi-Modal Comparative Evaluation
Executing the independent branch evaluations (Baseline) vs Late-Fusion.

In [ ]:
# Step 1: Run IMU-only baseline
%cd /content/RoadGuard/evaluation
!python eval_imu_branch.py --csv {NORMALIZED_CSV}

# Step 2: Run Vision-only baseline (using frames from Thessaloniki)
frame_dirs = glob.glob('/content/data/thessaloniki/**/frames', recursive=True)
FRAMES_DIR = frame_dirs[0] if frame_dirs else '/content/data/thessaloniki/frames'
!python eval_vision_branch.py --model /content/runs/roadguard_v1/weights/best.pt --frames {FRAMES_DIR}

# Step 3: Run Late-Fusion integration
!python eval_late_fusion.py
!python generate_report.py
%cd /content
print('Comparative evaluation reports generated.')

## 5. Federated Learning & Privacy Analysis
Simulating FL convergence and Differential Privacy tradeoffs for the thesis results chapter.

In [ ]:
# Running Personalized Fusion simulation with DP tradeoff logic
%cd /content/RoadGuard/evaluation
!python fl_personalized_fusion.py
%cd /content
print('FL and DP analysis completed.')

## 6. Final Results & Synthesis
Aggregating all visualizations for the experimental results section.

In [ ]:
print('--- SYNERGY ANALYSIS: LATE FUSION VS SINGLE BRANCHES ---')
try:
    display(Image('/content/RoadGuard/evaluation/results/comparison_chart.png'))
except:
    print('Fusion chart not found.')

print('\n--- PRIVACY ANALYSIS: EPSILON VS UTILITY ---')
try:
    display(Image('/content/RoadGuard/evaluation/results/fl_dp_tradeoff.png'))
except:
    print('DP tradeoff chart not found.')

print('\n--- FEDERATED COLLABORATION CHART ---')
try:
    display(Image('/content/RoadGuard/evaluation/results/fl_full_comparison.png'))
except:
    print('FL comparison chart not found.')

print('\n--- FINAL METRICS TABLE ---')
try:
    df_table = pd.read_csv('/content/RoadGuard/evaluation/results/fl_comparison_table.csv')
    print(df_table.to_string(index=False))
except:
    print('Metrics table not found.')

In [ ]:
# 1. Sync all results to Google Drive for persistence
print('Syncing results to Google Drive...')
shutil.copytree('/content/RoadGuard/evaluation/results', f'{SAVE_DIR}/results', dirs_exist_ok=True)
if os.path.exists('/content/yolov8n_pothole.tflite'):
    shutil.copy('/content/yolov8n_pothole.tflite', f'{SAVE_DIR}/yolov8n_pothole.tflite')

# 2. Create a ZIP package for easy local download
print('Packaging artifacts for local download...')
PACKAGE_PATH = '/content/RoadGuard_Thesis_Package'
os.makedirs(PACKAGE_PATH, exist_ok=True)
shutil.copytree('/content/RoadGuard/evaluation/results', f'{PACKAGE_PATH}/results', dirs_exist_ok=True)
if os.path.exists('/content/yolov8n_pothole.tflite'):
    shutil.copy('/content/yolov8n_pothole.tflite', f'{PACKAGE_PATH}/yolov8n_pothole.tflite')

shutil.make_archive('/content/RoadGuard_Final_Results', 'zip', PACKAGE_PATH)

# 3. Trigger Download
from google.colab import files
files.download('/content/RoadGuard_Final_Results.zip')

print('\nDone! Download the ZIP and extract it to get all thesis figures and the Android model.')